# Aggregating Interactions by Interest Subtype

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle
import tqdm
import os
from collections import defaultdict
from Xenium_5K_IPF_constants import EPITHELIAL_SUBTYPE, WORK_PATH

# Option
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

# Constant Declaration
top_n = 10
top_n_sub = 10

# Select interest cell group
interest_label = "Epithelial" 

In [ ]:
pathway_dir = f"{WORK_PATH}Interaction/interaction_scores_{interest_label}"
celltype = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
sig_pm=pd.read_csv(f"{WORK_PATH}CSV/031_significant_pathway_in_{interest_label}_nei_to_int.csv", index_col="Pathway")

In [ ]:
# Subtype coding
subtypes_cat = pd.Categorical(celltype["subtype"])
subtype_codes_per_cell = subtypes_cat.codes  # shape = (n_cells,)
unique_subtypes = list(subtypes_cat.categories)
n_cells = len(celltype)
n_subtypes = len(unique_subtypes)

# Source_Subtype × Target_Subtype boolern matrix
allowed = np.zeros((n_subtypes, n_subtypes), dtype=bool)
for src, g in sig_pm.groupby("Source_Subtype"):
    s_code = subtypes_cat.categories.get_loc(src)
    # unique target coding
    for tgt in g["Target_Subtype"].unique():
        t_code = subtypes_cat.categories.get_loc(tgt)
        allowed[s_code, t_code] = True

In [ ]:
# --- main ---
def process_one_pathway(sig_p):
    with open(os.path.join(pathway_dir, f"{sig_p}.pkl"), "rb") as f:
        obj = pickle.load(f)
    d = obj["nei_to_int"]   # {(s, t): value}
    if not d:
        return sig_p, {}

    # (s, t, val) array（for High speed and vectorization）
    # keys()→(s, t)、values()→val 
    st_pairs = list(d.keys())
    s_idx = np.fromiter((p[0] for p in st_pairs), dtype=np.int64)
    t_idx = np.fromiter((p[1] for p in st_pairs), dtype=np.int64)
    vals  = np.fromiter(d.values(), dtype=np.float64)

    # subtype code of each (s, t)
    s_code = subtype_codes_per_cell[s_idx]
    t_code = subtype_codes_per_cell[t_idx]

    # boolearn mask
    m = allowed[s_code, t_code]
    if not m.any():
        return sig_p, {}

    # Mask and weight → sum for each t
    w = vals * m
    agg_by_t = np.bincount(t_idx, weights=w, minlength=n_cells)  # shape=(n_cells,)

    # Convert to dict
    nz = np.nonzero(agg_by_t)[0]
    out = {int(t): float(agg_by_t[t]) for t in nz}
    return sig_p, out

In [ ]:
subtype_array = {}
sig_pathway = list(set(sig_pm.index))

for sig_p in tqdm.tqdm(sig_pathway):
    k, out = process_one_pathway(sig_p)
    subtype_array[k] = defaultdict(float, out)

In [ ]:
# etract Epithelial cells
int_cell_count = celltype.loc[celltype["cell_type"]==interest_label, "subtype"].value_counts().to_dict()

In [ ]:
# epithelial subtypes × interactions score matrix 
subtype_pathway_score = pd.DataFrame(np.zeros((len(EPITHELIAL_SUBTYPE), len(sig_pathway)), dtype=float), index=EPITHELIAL_SUBTYPE , columns=sig_pathway)
# epithelial subtypes × interactions count(%) matrix 
subtype_pathway_count = subtype_pathway_score.copy() 
for sig_p in tqdm.tqdm(sig_pathway):
    for i, val in subtype_array[sig_p].items():
        subtype_pathway_score.loc[celltype["subtype"].iloc[i], sig_p] += val/int_cell_count[celltype["subtype"].iloc[i]]
        subtype_pathway_count.loc[celltype["subtype"].iloc[i], sig_p] += 100/int_cell_count[celltype["subtype"].iloc[i]]

In [ ]:
# top interaction
score_top_df = subtype_pathway_score.sum().sort_values(ascending=False).head(top_n)
count_list = pd.Series([subtype_pathway_count.mean()[i] for i in list(score_top_df.index)], index=score_top_df.index)
score_top_df.index = [f"{sig_pm.loc[n]["pathway_name"].iloc[0]}: {n}" for n in list(score_top_df.index)]
count_list.index = score_top_df.index

In [ ]:
# sum of each subtype
n_max_s = 0
for e in EPITHELIAL_SUBTYPE:
    score_top_df_s = subtype_pathway_score.loc[e].sort_values(ascending=False).head(top_n_sub)
    count_list_s = pd.Series([subtype_pathway_count.loc[e][i] for i in list(score_top_df_s.index)], index=score_top_df_s.index)
    n_max_s = max(n_max_s, count_list_s.values.max())
    
fig, axes = plt.subplots(4,2, figsize=(8.5,11))
axes = axes.flatten()

norm = matplotlib.colors.Normalize(vmin=0, vmax=n_max_s)
cmap = matplotlib.colormaps.get_cmap("Blues")
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)

order = score_top_df.sort_values(ascending=True)
labels = order.index
vals   = order.values
count_vals = count_list.loc[labels].to_numpy() 

axes[0].barh(labels, vals, color=sm.to_rgba(count_vals), edgecolor="black")
axes[0].spines["right"].set_visible(False)
axes[0].spines["top"].set_visible(False)
axes[0].set_title("Epithelial", fontsize=13)
axes[0].tick_params("y", labelsize=12)


for e, ax in zip(EPITHELIAL_SUBTYPE, axes[1:]): 
    score_top_df_s = subtype_pathway_score.loc[e].sort_values(ascending=False).head(top_n_sub)
    count_list_s = pd.Series([subtype_pathway_count.loc[e][i] for i in list(score_top_df_s.index)], index=score_top_df_s.index)
    score_top_df_s.index = [f"{sig_pm.loc[n]["pathway_name"].iloc[0]}: {n}" for n in list(score_top_df_s.index)]
    count_list_s.index = score_top_df_s.index

    order_s = score_top_df_s.sort_values(ascending=True)
    labels_s = order_s.index
    vals_s   = order_s.values
    count_vals_s = count_list_s.loc[labels_s].to_numpy() 
    
    ax.barh(labels_s, vals_s, color=sm.to_rgba(count_vals_s), edgecolor="black")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.tick_params("y", labelsize=12)
    ax.set_title(e, fontsize=13)
    ax.set_xlabel("")

cbar_ax = fig.add_axes([0.75, 0.03, 0.2, 0.016])

cbar = fig.colorbar(sm, cax=cbar_ax, ticks=[0, 20, 40, 60, 80], orientation="horizontal")
cbar_ax.tick_params("x", labelsize=12)
cbar_ax.set_title("Proportion of cells (%)", fontsize=13)
cbar_ax.yaxis.set_label_position("left")
axes[-1].set_xlabel("Signal score", fontsize=13)
axes[-2].set_xlabel("Signal score", fontsize=13)

fig.subplots_adjust(left=0.32,right=0.97,top=0.97,bottom=0.13, wspace=2, hspace=0.5)

for ax in axes:
    ticks = list(ax.get_xticks())
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(int(n/1000))+"K" for n in ticks], fontsize=12)

fig.savefig(f"{WORK_PATH}Figure/032_Epithelial_interaction_Top_{top_n}.png", dpi=500, format="png", bbox_inches=None)

# Cell-cell Interaction along Pseudo-temporal Contiuum

In [ ]:
import pandas as pd
import numpy as np
import scipy
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import tqdm
import pickle
import os
import glob
from statsmodels.stats.multitest import multipletests
from Xenium_5K_IPF_constants import WORK_PATH

# Option
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl


In [ ]:
pathway_dir = f"{WORK_PATH}Interaction/interaction_scores_Epithelial" 
celltype = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv") 
cellchat_db_f=pd.read_pickle(f"{WORK_PATH}Interaction/interaction_data.pkl")
position_df=pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")

In [ ]:
bins = 10

In [ ]:
celltype_c = celltype.copy()
celltype_c["index"]=celltype_c.index
celltype_c.set_index("cell_id", inplace=True)
position_df_m = pd.merge(left=position_df, right=celltype_c[["index"]], left_index=True, right_index=True, how="left")

In [ ]:
position_df_m["CP_bins"]=pd.cut(position_df_m["C_position"], bins, labels=False)
position_df_m["CP_bins"]=pd.Categorical(position_df_m["CP_bins"], ordered=True)

In [ ]:
# {interest cell index: CP_bins}
index_tbins_dict = defaultdict(int)
for p_df in position_df_m.itertuples():
    index_tbins_dict[p_df[4]]=(p_df[5])

In [ ]:
# each CP_bins cell count
cp_bins_counts = position_df_m["CP_bins"].value_counts().to_dict()

In [ ]:
results_df = {
    "nei_to_int":defaultdict(lambda: defaultdict(float)),
    "int_to_nei":defaultdict(lambda: defaultdict(float))
}

valid_indices = set(position_df_m["index"]) 

for fpath in tqdm.tqdm(sorted(glob.glob(os.path.join(pathway_dir, "*.pkl")))):
    int_name = os.path.splitext(os.path.basename(fpath))[0]
    
    with open(fpath, "rb") as f:
        obj = pickle.load(f)  # {"direction":{(send_idx, reci_idx):values}
    
    obj_n2i = obj["nei_to_int"]  # (send_idx, reci_idx):values
    obj_i2n = obj["int_to_nei"]

    for (_, i), score in obj_n2i.items():
        if i not in valid_indices:
            continue
        cp_bin =index_tbins_dict[i]
        results_df["nei_to_int"][f"{int_name}"][cp_bin] += score/cp_bins_counts[cp_bin]

    for (i, _), score in obj_i2n.items():
        if i not in valid_indices:
            continue
        cp_bin =index_tbins_dict[i]
        results_df["int_to_nei"][f"{int_name}"][cp_bin] += score/cp_bins_counts[cp_bin]


In [ ]:
merge_df = [
    pd.DataFrame({path: dict(val) for path, val in results_df["nei_to_int"].items()}).sort_index().fillna(0),
    pd.DataFrame({path: dict(val) for path, val in results_df["int_to_nei"].items()}).sort_index().fillna(0)
]

In [ ]:
for m_df,t in zip(merge_df, ["n2i", "i2n"]):
    csv = m_df.loc[:,m_df.sum().sort_values(ascending=False).index].T.rename(columns={n:f"bin_{n}" for n in range(bins)})
    csv["sum"] = csv.sum(axis=1)
    csv.to_csv(f"{WORK_PATH}CSV/032_interaction_cPosition_rank_{t}.csv", index=True)

In [ ]:
top_p = 50
merge_df_top=[m_df.loc[:,m_df.sum().sort_values(ascending=False).head(top_p).index] for m_df in merge_df]

In [ ]:
for n in range(2):
    merge_df_top[n] = (merge_df_top[n]-merge_df_top[n].min())/(merge_df_top[n].max()-merge_df_top[n].min())
    merge_df_top[n].columns = [cellchat_db_f.set_index("interaction_name").loc[c, "pathway_name"]+": "+c for c in merge_df_top[n].columns]

In [ ]:
d = 0

if d==0:
    cluster_names = {
        1: 'Early',
        2: 'Late',
        3: 'Intermediate',
    }
    tl = "Neighbor -> Epithelial" 
    col_thr = 3.8
    save_txt = "nei_to_int"
    
else:
    cluster_names = {
        1: 'Intermediate',
        2: 'Late',
        3: "Very early",
        4: "Early"
    }
    tl = " Epithelial -> Neighbor" 
    col_thr = 3.5
    save_txt = "int_to_nei"


position_color_dict = {
    "Early": "dodgerblue", 
    "Late": "orangered",
    "Intermediate":"mediumorchid",
    "Very early":"green"
}
position_color = [position_color_dict[c] for c in cluster_names.values()]
scipy.cluster.hierarchy.set_link_color_palette(position_color)

In [ ]:
g = sns.clustermap(
    merge_df_top[d].T,
    row_cluster=True,
    col_cluster=False,
    cmap="viridis",
    method="ward",
    cbar_kws={"label": "Normalized signal (0-1)"}
)

In [ ]:
# linkage
Z = g.dendrogram_row.linkage
row_order = g.dendrogram_row.reordered_ind
labels = merge_df_top[d].T.index[row_order]

fig = plt.figure(figsize=(7, 11))
plt.subplots_adjust(left=0, right=1, top=0.89, bottom=0.03)
gs = gridspec.GridSpec(1, 3, width_ratios=[3, 6, 7], wspace=0, hspace=0.08)

# dendrogram
ax0 = plt.subplot(gs[0])
dnd = scipy.cluster.hierarchy.dendrogram(
    Z,
    labels=labels,
    orientation="left",
    leaf_rotation=0,
    leaf_font_size=10,
    ax=ax0,
    color_threshold=col_thr,
    above_threshold_color="grey",
)
ax0.axis("off")

# heatmap
ax1 = plt.subplot(gs[1])
sns.heatmap(
    merge_df_top[d].T.iloc[row_order],
    cmap="viridis",
    cbar_kws={"label": "Normalized signal (0-1)"},
    ax=ax1,
    cbar=False,
    linecolor="black",
    linewidths=0,
)

ax1.yaxis.tick_right()
ax1.yaxis.set_label_position("right")
ax1.set_yticklabels(labels, rotation=0)
ax1.set_xticks([0,bins/2, bins], labels=[0.0, 0.5, 1.0], rotation=0, fontsize=11)
ax1.tick_params("x", top=True, labeltop=False, bottom=True, labelbottom=True)
ax1.tick_params("y", labelsize=12)
ax0.invert_yaxis()

# colorbar setting
norm = matplotlib.colors.Normalize(vmin=0, vmax=1)
center_ax = [sum(gs.get_width_ratios()[0:1])/sum(gs.get_width_ratios()), sum(gs.get_width_ratios()[0:2])/sum(gs.get_width_ratios())]
sm_cp = plt.cm.ScalarMappable(cmap="plasma", norm=norm)
sm_val = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
cbar_ax_cp = fig.add_axes((center_ax[0], 0.9, center_ax[1]-center_ax[0], 0.025))
cbar_ax_val = fig.add_axes((center_ax[1]+0.1, 0.92, 1-center_ax[1]-0.3, 0.015))
cbar_cp = plt.colorbar(sm_cp, cax=cbar_ax_cp, ticks=[0.0, 0.5, 1.0], orientation="horizontal")
cbar_val = plt.colorbar(sm_val, cax=cbar_ax_val, ticks=[0.0, 1.0], orientation="horizontal")
cbar_cp.ax.set_xticklabels(['0.0', '0.5', '1.0'], fontsize=11)
cbar_val.ax.set_xticklabels(['min', "max"], fontsize=11)

cbar_cp.ax.set_ylabel("Continuum\nposition", fontsize=13, rotation=0, va="center", ha="right", weight=800, multialignment='center', labelpad=5)
cbar_val.ax.set_title("Normalized\nsignal score", fontsize=12)

cbar_ax_cp.tick_params("x", top=True, labeltop=True, bottom=False, labelbottom=False)
cbar_ax_cp.set_yticks([0.5], labels=[""])
cbar_ax_cp.tick_params("y", size=8)

# title setting
ti_ax = fig.add_axes((center_ax[0]-0.1, 0.95, center_ax[1]-center_ax[0]+0.2, 0.05))
ti_ax.text(s=tl, x=0.5, y=0.5, ha="center", va="center", fontsize=17)
ti_ax.axis("off")

# clustering annotation
cluster_assignments = scipy.cluster.hierarchy.fcluster(Z, t=col_thr, criterion='distance')
clusters_in_order = cluster_assignments[row_order]

# ranges of clusters
cluster_ranges = {}
for idx, (cluster_id, c) in enumerate(zip(clusters_in_order, dnd['leaves_color_list'])):
    if cluster_id not in cluster_ranges:
        cluster_ranges[cluster_id] = [idx, idx, c]
    else:
        cluster_ranges[cluster_id][1] = idx

# add cluster label
ax2 = plt.subplot(gs[2], sharey=ax1)
ax2.set_xlim(0, 1.01) 
ax2.axis("off")   

ax1.set_zorder(2)  
ax2.set_zorder(1) 
dnd['leaves_color_list']
mid = 0.87
for cluster_id, (start, end, c) in cluster_ranges.items():
    y_center = (start + end + 1) / 2

    r1 = matplotlib.patches.Rectangle(xy=(0, start), width=mid, height=end-start+1, ec=None, fill="grey", fc=c, alpha=0.3)
    r2 = matplotlib.patches.Rectangle(xy=(mid, start), width=1-mid, height=end-start+1, ec=None, fill="grey", fc=c, alpha=1)
    
    ax2.plot([0, 1], [start, start], color="black", lw=1, clip_on=False)
    ax2.add_patch(r1)
    ax2.add_patch(r2)
\
    ax2.text(
        (1+mid)/2, y_center+0.1,
        f"{cluster_names[cluster_id]}",
        va="center", ha="center", fontsize=18,
        color="white", weight=700, rotation=90
    )
ax2.plot([0, 0], [0, len(labels)], color="black", lw=1, clip_on=False)
ax2.plot([1, 1], [0, len(labels)], color="black", lw=1, clip_on=False)
ax2.plot([0, 1], [len(labels), len(labels)], color="black", lw=1, clip_on=False)

plt.savefig(f"{WORK_PATH}Figure/032_Interaction_Cposition_{save_txt}.png", dpi=500, format="png", bbox_inches=None)
plt.show()

# Directionality of Pathways Over a Specific Continuum Period

In [ ]:
import pandas as pd
import numpy as np
import scipy
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import tqdm
import pickle
import scanpy as sc
import glob
import os
from pycirclize import Circos
from pycirclize.utils import ColorCycler
from statsmodels.stats.multitest import multipletests
from Xenium_5K_IPF_constants import EPITHELIAL_SUBTYPE, ALL_SUBTYPE_LIST, TITLE_ABBREVIATION, MAJOR_CLUSTER_COLORS, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl


In [ ]:
pathway_dir = f"{WORK_PATH}Interaction/interaction_scores_Epithelial" 
celltype = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv") 
cellchat_db_f=pd.read_pickle(f"{WORK_PATH}Interaction/interaction_data.pkl")
position_df=pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")


In [ ]:
subtype_to_major = celltype.set_index("subtype")["cell_type_major"].to_dict()

In [ ]:
celltype_c = celltype.copy()
celltype_c["index"]=celltype_c.index
celltype_c.set_index("cell_id", inplace=True)
position_df_m = pd.merge(left=position_df, right=celltype_c[["index"]], left_index=True, right_index=True, how="left")

In [ ]:
cp_bin_n = 5

In [ ]:
def Cp_bin(x):
    if x == 1:
        return cp_bin_n-1
    else:
        return int((cp_bin_n*x)//1)
position_df_m['Cp_bin'] = position_df_m['C_position'].map(lambda x:Cp_bin(x))
position_df_m["Cp_bin"] = pd.Categorical(position_df_m["Cp_bin"], categories=[n for n in range(cp_bin_n)], ordered=True)

In [ ]:
cp_bins_counts = pd.crosstab(position_df_m["Cp_bin"], position_df_m["subtype"])
cp_bins_counts_rt = cp_bins_counts.div(cp_bins_counts.sum(axis=1), axis=0)
cp_bins_counts_rt

In [ ]:
# Nei to Epi

# int_pathway ="TGFB1 - ACVR1B_TGFbR2"
# int_pathway ="TGFB2 - ACVR1B_TGFbR2"
# int_pathway ="TGFB3 - ACVR1B_TGFbR2"
# int_pathway ="POSTN - ITGAV_ITGB3"
# int_pathway ="POSTN - ITGAV_ITGB5"

to = "nei_to_int"

In [ ]:
# Epi to Nei

# int_pathway ="PDGFA - PDGFRA"
# int_pathway ="PDGFA - PDGFRB"
int_pathway = "VEGFA - KDR"
# int_pathway ="C5 - C5AR1"

to = "int_to_nei"

In [ ]:
n_cells_cutoff = 0.03

epi_to_i = {k: i for i, k in enumerate(EPITHELIAL_SUBTYPE)}
all_to_i = {k: i for i, k in enumerate(ALL_SUBTYPE_LIST)}

int_cp_bins_arr = {
    cp: np.zeros((len(ALL_SUBTYPE_LIST), len(EPITHELIAL_SUBTYPE)), dtype=float)
    for cp in range(cp_bin_n)
}

subtype_map = celltype_c.set_index("index")["subtype"].to_dict()
cp_map = position_df_m.set_index("index")["Cp_bin"].to_dict()

with open(os.path.join(pathway_dir, f"{int_pathway}.pkl"), "rb") as f:
    obj = pickle.load(f)[to]

if to == "nei_to_int":
    for (s, r), score in obj.items():
        cp_bin = cp_map.get(r)
        if cp_bin is None:
            continue
    
        sen_type = subtype_map[s]
        rec_type = subtype_map[r]
        if cp_bins_counts_rt.loc[cp_bin, rec_type] > n_cells_cutoff:
            int_cp_bins_arr[cp_bin][all_to_i[sen_type], epi_to_i[rec_type]] += score/cp_bins_counts.loc[cp_bin,rec_type]
        else:
            int_cp_bins_arr[cp_bin][all_to_i[sen_type], epi_to_i[rec_type]] += 0
            
else:  # "int_to_nei"
    for (s, r), score in obj.items():
        cp_bin = cp_map.get(s)
        if cp_bin is None:
            continue
    
        sen_type = subtype_map[s]
        rec_type = subtype_map[r]
        if cp_bins_counts_rt.loc[cp_bin, sen_type] > n_cells_cutoff:
            int_cp_bins_arr[cp_bin][all_to_i[rec_type], epi_to_i[sen_type]] += score/cp_bins_counts.loc[cp_bin, sen_type]
        else:
            int_cp_bins_arr[cp_bin][all_to_i[rec_type], epi_to_i[sen_type]] += 0

int_cp_bins = {
    cp: pd.DataFrame(
        int_cp_bins_arr[cp],
        index=ALL_SUBTYPE_LIST,
        columns=EPITHELIAL_SUBTYPE
    )
    for cp in range(cp_bin_n)
}

cut_bin_type = (cp_bins_counts_rt <= n_cells_cutoff).stack()
cut_bin_type = cut_bin_type[cut_bin_type].index.tolist()

In [ ]:
records = []
global_max = max(arr.max() for arr in int_cp_bins_arr.values())

for cp_bin, arr in int_cp_bins_arr.items():
    n_row, n_col = arr.shape

    for i in range(n_row):
        for j in range(n_col):
            score = arr[i, j]
            records.append({
                "cp_bin": cp_bin,
                "ALL_idx": int(i),
                "EPI_idx": int(j),
                "score": score,
                "scaled_global": score / global_max if global_max > 0 else 0.0,
            })

df_int = pd.DataFrame(records)


In [ ]:
df_nz = df_int[df_int["score"] > 0].copy()
pct = 5
thr = df_nz["score"].quantile(1-pct/100) 
top_df = df_nz[df_nz["score"] >= thr].sort_values("score", ascending=False)
top_df["Nei_type"] = top_df["ALL_idx"].map(lambda x: subtype_to_major[ALL_SUBTYPE_LIST[x]])
top_df["Epi_subtype"] = top_df["EPI_idx"].map(lambda x: EPITHELIAL_SUBTYPE[x])

In [ ]:
n_epi = len(EPITHELIAL_SUBTYPE)
n_all = len(ALL_SUBTYPE_LIST)

sectors_name = [f"Epi\n[{n/cp_bin_n} - {(n+1)/cp_bin_n}]" for n in range(cp_bin_n)]+["Nei"] 
sectors_subtype_n = [n_epi]*cp_bin_n+[n_all]
epi_sub_list = [list(np.arange(0,n_epi, 1))]*cp_bin_n

for cp, ty in cut_bin_type:
    sectors_subtype_n[cp] -= 1
    epi_sub = epi_sub_list[cp].copy()
    rm = EPITHELIAL_SUBTYPE.index(ty)
    epi_sub.remove(rm)
    epi_sub_list[cp] = epi_sub.copy()

sectors = {name:l for name,l in zip(sectors_name, sectors_subtype_n)}

int_spa_major = 25
int_spa_minor = (180-int_spa_major)*(n_all-sum(sectors_subtype_n[:-1]))/(4*n_all)
spaces = [int_spa_minor] * 4 + [int_spa_major]*2

In [ ]:
top_df_new = []
for n,e in enumerate(epi_sub_list):
    if len(e) < n_epi:
        temp_df = top_df[top_df["cp_bin"]==n].copy()
        temp_df["EPI_idx"] = temp_df["EPI_idx"].map(lambda x: e.index(x))
        top_df_new.append(temp_df)
    else:
        top_df_new.append(top_df[top_df["cp_bin"]==n].copy())
top_df_new = pd.concat(top_df_new)

In [ ]:
circos = Circos(sectors, space=spaces, start=-360+int_spa_major/2, end=int_spa_major/2)

sm_cp = plt.get_cmap("plasma",cp_bin_n)
sm_epi = plt.get_cmap("Set3",12)
sm_epi_list = [matplotlib.colors.rgb2hex(sm_epi(i))+"ff" for i in range(sm_epi.N)]
sm_epi_col = [[sm_epi_list[n] for n in epi_sub_list[m]] for m in range(cp_bin_n)]

bar_max=0
for n in range(cp_bin_n):
    if sum(top_df_new["cp_bin"]==n) >0:
        bar_max = max(pd.DataFrame({"score":top_df_new[top_df_new["cp_bin"]==n].groupby("EPI_idx")["score"].sum()},index=np.arange(0,n_epi,1))["score"].max(), bar_max)
    
if bar_max > 8000:
    bar_max = ((bar_max//5000)+1)*5000
elif bar_max > 2000:
    bar_max = ((bar_max//1000)+1)*1000
elif bar_max > 1000:
    bar_max = ((bar_max//200)+1)*200
else:
    bar_max = 1000

arrow_rad = []
    
for m,sector in enumerate(circos.sectors):
    # subtype colors
    track_subcol = sector.add_track((45, 50))
    track_subcol.axis()
    
    # subtype names
    track_subtype = sector.add_track((51, 51))
    track_subtype.axis()
    
    if sector.name.startswith("Ep"):
        sub_list = [EPITHELIAL_SUBTYPE[n] for n in epi_sub_list[m]]
    elif sector.name.startswith("Ne"):
        sub_list = ALL_SUBTYPE_LIST[::-1]
    
    for i in range(int(track_subtype.size)):
        start, end = i, i + 1

        track_subtype.axis(fc="white", ec="white")
        track_subtype.text(sub_list[i], (end + start) / 2, size=12)

        if sector.name.startswith("Ep"):
            track_subcol.rect(start, end, fc=sm_epi(epi_sub_list[m][start]))
        elif sector.name.startswith("Ne"):
            track_subcol.rect(start, end, fc=MAJOR_CLUSTER_COLORS[subtype_to_major[ALL_SUBTYPE_LIST[n_all-start-1]]])

    if sector.name.startswith("Ep"): 
        # Bar Track
        track_bar = sector.add_track((72, 92), r_pad_ratio=0.1)
        track_bar.axis()
        track_bar.grid(y_grid_num=3, color="#555555", ls="--", lw=0.8)
        df_bin = top_df_new[top_df_new["cp_bin"]==m]
        epi_score = pd.DataFrame({"score":df_bin.groupby("EPI_idx")["score"].sum()},index=np.arange(0,len(epi_sub_list[m]),1)).fillna(0)
        track_bar.bar(np.arange(0,len(epi_score),1)+0.5, epi_score["score"], vmax=bar_max, color=sm_epi_col[m], ec="grey")

        # Continuum position colors
        track_subcol = sector.add_track((93, 100))
        track_subcol.axis(fc=sm_cp(m))
        if m < cp_bin_n/2:
            track_subcol.text(sector.name.split("\n")[1],size=11, color="white", weight=500)
        else:
            track_subcol.text(sector.name.split("\n")[1],size=11, color="black", weight=400)
            
        if m == 0:
            track_bar.yticks(np.linspace(0, bar_max, 3), [0, int(bar_max/2), int(bar_max)], side="left", line_kws=dict(color="grey", lw=0.8))
            track_subcol.yticks([1], ["Continuum-position intervals"], vmin=0, vmax=2, side="left", line_kws=dict(lw=1), text_kws=dict(weight=500))

        else:
            track_bar.yticks(np.linspace(0, bar_max, 3), side="left", line_kws=dict(color="grey", lw=0.8))
    
        arrow_rad.append(sector.rad_lim)
    

for _,d in top_df_new.iterrows():
    if to == "nei_to_int":
        circos.link(("Nei", n_all-1-d["ALL_idx"], n_all-d["ALL_idx"]), (sectors_name[int(d["cp_bin"])], d["EPI_idx"]+1, d["EPI_idx"]),
                    direction=1, fc=MAJOR_CLUSTER_COLORS[d["Nei_type"]], alpha=0.3, ec="black", lw=1)
    else:        
        circos.link((sectors_name[int(d["cp_bin"])], d["EPI_idx"]+1, d["EPI_idx"]), ("Nei", n_all-1-d["ALL_idx"], n_all-d["ALL_idx"]), 
                    direction=1, fc=sm_epi_col[d["cp_bin"]][d["EPI_idx"]], alpha=0.3, ec="black", lw=1)


fig = circos.plotfig(figsize=(8.5,8))

ax = fig.axes[0]

# Arrows of continuum position intervals
for m in range(cp_bin_n-1):
    rad_int = arrow_rad[m+1][0]-arrow_rad[m][1]
    # arrow body
    theta = np.linspace(arrow_rad[m][1]+rad_int/8, arrow_rad[m+1][0]-rad_int/3.5, 300)
    r = np.full_like(theta, 96)
    ax.plot(theta, r, lw=6, color="#4a4a4a")
    
    # arrow head
    tri = plt.Polygon(((arrow_rad[m+1][0]-rad_int/3, 92),(arrow_rad[m+1][0]-rad_int/3,100),(arrow_rad[m+1][0]-rad_int/20,96)),fc="#4a4a4a")
    ax.add_patch(tri)

# modify
cx, cy = ax.transAxes.transform((0.5, 0.5))

for txt in ax.texts:
    label = txt.get_text()
    if label[:2] in ["[0", "Sc"]:
        continue
    if label[:2] in ["Co"]:
        txt.set_rotation(0)
        txt.set_size(12)
        continue
    if label[-1:] in ["0"]:
        txt.set_rotation(0)
        txt.set_size(10)
        continue
        
    rot = txt.get_rotation()
    if rot > 90:
        txt.set_rotation(rot + 90)
    else:
        txt.set_rotation(rot - 90)

    x_disp, y_disp = txt.get_transform().transform(txt.get_position())

    txt.set_va("center")
    txt.set_rotation_mode("anchor")

    if x_disp >= cx:
        # The label on the right is left-aligned.
        txt.set_ha("left")
    else:
        # The label on the leftt is right-aligned.
        txt.set_ha("right")
    if label in TITLE_ABBREVIATION:
        txt.set_text(TITLE_ABBREVIATION[label])
    if label[-2:-1] == "_":
        txt.set_text(label[:-2])

count_ax = fig.add_axes([0.12, 0.06, 0.05, 0.15])
count_ax.set(xlim=(-0.6, 0.6),ylim=(0, 105))
count_ax.spines["top"].set_visible(False)
count_ax.spines["right"].set_visible(False)
count_ax.set_xticks([])
count_ax.set_title(f"Top {pct}% contribution\nto total score (%)")
count_ax.set_yticks([0, 50, 100], [0, 50, 100], fontsize=12)
per = 100*top_df_new["score"].sum()/df_int["score"].sum()
count_ax.bar(x=0, height=per, color="grey", ec="black")

fig.text(s=f"{int_pathway}",x=0.18,y=0.85, size=16, va="bottom", ha="center", weight=800)
fig.text(s=f"(top {pct}% score)",x=0.18,y=0.85, size=15, va="top", ha="center")

if to == "nei_to_int":
    fig.text(s="Sender (neighbor)",x=0.08,y=0.5, size=14, rotation=90,  ha="left", va="center", weight=700)
    fig.text(s="Reciever (epithelial)",x=0.98,y=0.5, size=14, rotation=270, ha="right", va="center", weight=700)
else:
    fig.text(s="Reciever (neighbor)",x=0.08,y=0.5, size=14, rotation=90,  ha="left", va="center", weight=700)
    fig.text(s="Sender (epithelial)",x=0.98,y=0.5, size=14, rotation=270, ha="right", va="center", weight=700)

plt.savefig(f"{WORK_PATH}/Figure/032_interaction_along_CPinterval_{to}_{int_pathway}.png", dpi=300, format="png", bbox_inches=None)

# Downstream Analysis Between Signal Positive and Negative

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import squidpy as sq
import decoupler as dc
import gseapy as gp
import scipy
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import tqdm
import pickle
import os
import glob
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
from statsmodels.stats.multitest import multipletests
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, EPITHELIAL_SUBTYPE, WORK_PATH

os.makedirs(f"{WORK_PATH}Figure/032_Downstream_analysis/", exist_ok=True)

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl


In [ ]:
celltype = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv") 
cellchat_db_f=pd.read_pickle(f"{WORK_PATH}Interaction/interaction_data.pkl")

In [ ]:
interest_label = "Epithelial"     
pathway_dir = f"{WORK_PATH}Interaction/interaction_scores_Epithelial" 

In [ ]:
celltype_c = celltype.copy()
celltype_c["index"]=celltype_c.index
celltype_c.set_index("cell_id", inplace=True)

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/integrated_anndata.h5ad")
adata = adata[celltype_c.index,].copy()
adata.obs = pd.merge(adata.obs, celltype_c[["cell_type_major", "subtype"]], left_index=True, right_index=True)

In [ ]:
int_pathway ="TGFb"
# int_pathway ="PERIOSTIN"

interactions = cellchat_db_f[cellchat_db_f["pathway_name"]==int_pathway]["interaction_name"]
dir_name = int_pathway

rej = list(set(cellchat_db_f[cellchat_db_f["pathway_name"]==int_pathway]["gene_list"].sum()))

In [ ]:
interactions = ["C5 - C5AR1"]
# interactions = ["HGF - MET"]

dir_name = interactions[0]

assert os.path.isfile(f"{pathway_dir}/{dir_name}.pkl"), "Enter the interaction pair by the wrong name."

rej = list(set(cellchat_db_f[cellchat_db_f["interaction_name"].isin(interactions)]["gene_list"].sum()))

In [ ]:
fig_dir = f"{WORK_PATH}Figure/032_Downstream_analysis/{dir_name}/"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
results_path_df = {
    "nei_to_int":defaultdict(float),
    "int_to_nei":defaultdict(float)
}

for inter in tqdm.tqdm(interactions):
    with open(os.path.join(pathway_dir, f"{inter}.pkl"), "rb") as f:
        obj = pickle.load(f)  # {"direction":{(send_idx, reci_idx):values}

        obj_n2i = obj["nei_to_int"]  # (send_idx, reci_idx):values
        obj_i2n = obj["int_to_nei"]
        
        for (_, i), score in obj_n2i.items():
            results_path_df["nei_to_int"][i] += score
        
        for (i, _), score in obj_i2n.items():
            results_path_df["int_to_nei"][i] += score


In [ ]:
adata_m = adata[adata.obs["cell_type_major"]==interest_label].copy()

In [ ]:
sub_d = adata_m.obs["subtype"].unique()
sub_d = [s for s in ALL_SUBTYPE_LIST if s in sub_d]
adata_m.obs["subtype"] = pd.Categorical(adata_m.obs["subtype"], categories=sub_d, ordered=True)

In [ ]:
pathawy_score = {
    "nei_to_int":pd.DataFrame,
    "int_to_nei":pd.DataFrame
}

for direc in ["nei_to_int", "int_to_nei"]:
    pathawy_score[direc]=pd.DataFrame.from_dict(results_path_df[direc], orient="index", columns=[f"int_scores"])
    celltype_c[f"int_{direc}"] = pd.Categorical(celltype_c["index"].isin(pathawy_score[direc].index).map({True: "positive", False: "negative"}))
    adata_m.obs = pd.merge(left=adata_m.obs, right=celltype_c[celltype_c["cell_type"]==interest_label][[f"int_{direc}"]], left_index=True, right_index=True, how="inner")

In [ ]:
adata_m = adata_m[:,[g for g in adata.var_names if g not in rej]]

In [ ]:
ttt = "int_nei_to_int"

In [ ]:
ttt = "int_int_to_nei"

In [ ]:
adata_m.obs = adata_m.obs.rename(columns={ttt: "Signal"})

In [ ]:
pdata = dc.pp.pseudobulk(
    adata=adata_m,
    sample_col="sample",
    groups_col=["subtype", "Signal"] ,
    mode="sum",
)

In [ ]:
pdata.obs["Signal"] = pd.Categorical(pdata.obs["Signal"], categories=["positive", "negative"], ordered=True)
pdata.obs["subtype"] = pd.Categorical(pdata.obs["subtype"], categories=EPITHELIAL_SUBTYPE, ordered=True)

In [ ]:
dc.pl.filter_samples(
    adata=pdata,
    groupby=["sample", "subtype", "Signal"],
    min_cells=10,
    min_counts=1000,
    figsize=(5, 8),
)
plt.savefig(f"{fig_dir}nei2epi_signal_QC.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
dc.pp.filter_samples(pdata, min_cells=10, min_counts=1000)

In [ ]:
plt.figure(figsize=(5,3))
plt.subplots_adjust(left=0.3, right=0.6, top=0.98, bottom=0.15)
sns.countplot(
    data=adata_m.obs,
    y="subtype",
    hue="Signal",
    hue_order=["positive", "negative"],
    palette={"positive": "teal", "negative": "#aaaaaa"},  
    legend=False,
)
plt.legend(["positive", "negative"], title=f"{dir_name} signal", fontsize=11, title_fontsize=11, loc='center left', bbox_to_anchor=(1, 0.5))
plt.ylabel("")
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.tick_params("y", labelsize=12)
plt.xlabel("Count", fontsize=12)

plt.show()

plt.savefig(f"{fig_dir}nei2epi_signal_count.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
sub_adata_list = [pdata[pdata.obs["subtype"]==e] for e in sub_d]

In [ ]:
def DESeq2_func(data):
    # Build DESeq2 object
    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet(
        adata=data,
        design_factors=["Signal", "sample"],
        refit_cooks=True,
        inference=inference,
    )
    
    # Compute LFCs
    dds.deseq2()
    
    # Extract contrast between conditions
    stat_res = DeseqStats(dds, contrast=["Signal", "positive", "negative"], inference=inference)
    
    # Compute Wald test
    stat_res.summary()
    
    return stat_res.results_df

In [ ]:
hoge = gp.get_library(f"{WORK_PATH}h.all.v2025.1.Hs.symbols.gmt")
emt_set = {"EPITHELIAL_MESENCHYMAL_TRANSITION":hoge["HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"]}

In [ ]:
pre_res_list = []
for sub in sub_adata_list:
    res = DESeq2_func(sub)    
    rank_sub = (
        res[['stat']]
        .dropna()
        .squeeze()
        .sort_values(ascending=False)
    )
    rank_sub = rank_sub.loc[~rank_sub.index.duplicated(keep=False)].copy()
    
    pre_res = gp.prerank(
    rnk=rank_sub,
    gene_sets=emt_set,
    processes=4,
    permutation_num=5000,
    min_size=15,
    max_size=500,
    outdir=None,
    seed=42,
    )
    pre_res_list.append(pre_res.res2d.iloc[0])
emt_df = pd.DataFrame({key:val for key, val in zip(sub_d, pre_res_list)}).T

In [ ]:
fuga = emt_df[["NES", "NOM p-val"]].astype(float).copy()

In [ ]:
fuga["log10 p-val"] = -np.log10(fuga["NOM p-val"].clip(1e-3, 1))

In [ ]:
# HGF-MET
adtxt = {
    "AT1":["left", "top", (-0.1, -0.1)],
    "RAS":["left", "bottom", (-0.05, 0.05)],
}
tit = "HGF-MET"

In [ ]:
# TGFb
adtxt = {
    "AT1":["right", "bottom", (0.15, 0.05)],
    "KRT5-/KRT17+":["left", "top", (0, 0)],
    "Basal":["left", "bottom", (-0.15, 0.1)],
    "Ciliated":["right", "top", (0, 0)],
}
tit = "TGFb"

In [ ]:
fig,ax=plt.subplots(figsize=(4,3))
ax.set_xlim(-3, 3)
ax.set_ylim(0,3.2)
ax.set_xlabel("NES",fontsize=13)
ax.set_ylabel("$-\log_{10}{(P \; \mathrm{value})}$", fontsize=13)
for i, row in fuga.iterrows():
    nes = row["NES"]
    pvl = row["log10 p-val"]
    if pvl >= -np.log10(0.05):
        if nes > 0:
            ax.scatter(y=pvl, x=nes, c="red", edgecolor="black", alpha=0.7)
        else:
            ax.scatter(y=pvl, x=nes, c="blue", edgecolor="black", alpha=0.7)
    else:
        ax.scatter(y=pvl, x=nes, c="grey", edgecolor="black", alpha=0.7)

    marg = 0.1
    if i in adtxt:
        v = adtxt[i]
        if v[0] == "right":
            marg = -marg
        ax.text(y=pvl+v[2][1], x=nes+v[2][0]+marg, s=i, fontsize=12, ha=v[0], va=v[1])
    else:
        ax.text(y=pvl, x=nes+marg, s=i, fontsize=12)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axhline(y=-np.log10(0.05), c="grey", ls="--")
ax.axvline(x=0, c="grey", ls="--")
ax.set_yticks(ticks=[0, 1, 2, 3], labels=["0", "1", "2", "3<"], fontsize=12)
ax.set_title(f"{tit}", weight=500)


fig.tight_layout()

plt.savefig(f"{fig_dir}nei2epi_emt_volcano.png", dpi=500, format="png", bbox_inches=None)

## each subtype comparation

In [ ]:
sub_adata_list.append(pdata)
sub_d.append(interest_label)

In [ ]:
nnn = 2
sub_d[nnn]

In [ ]:
pdata_s = sub_adata_list[nnn].copy()

In [ ]:
results_df = DESeq2_func(pdata_s)    

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", figsize=(3.5,3.5), top=10)
plt.ylabel("$-\log_{10}{(P \; \mathrm{value})}$", fontsize=12)
plt.xlabel("$\log_{2}{(\mathrm{fold \; change})}$", fontsize=12)
plt.title(f"{dir_name} signal\npositive vs. negative\n{sub_d[nnn]}", fontsize=12)
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.savefig(f"{fig_dir}nei2epi_signal_volcano_{sub_d[nnn].replace("/","_")}.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
rank_s = (
    results_df[['stat']]
    .dropna()
    .squeeze()
    .sort_values(ascending=False)
)
rank_s = rank_s.loc[~rank_s.index.duplicated(keep=False)].copy()

In [ ]:
got = [
    'MSigDB_Hallmark_2020',
    'KEGG_2019_Mouse',
    'KEGG_2021_Human',
    'Reactome_Pathways_2024',
    'GO_Biological_Process_2025',
    'GO_Molecular_Function_2025',
    f"{WORK_PATH}h.all.v2025.1.Hs.symbols.gmt"
]

ggg = 6

In [ ]:
prerank_res = gp.prerank(
    rnk=rank_s,
    gene_sets=got[ggg],
    processes=4,
    permutation_num=5000,
    min_size=15,
    max_size=500,
    outdir=None,
    seed=42,
)

In [ ]:
df_q = prerank_res.res2d
df_q = df_q[df_q["FDR q-val"]<0.05]
df_q = df_q[["Term", "NES", "FDR q-val"]]
df_q["Term"] = [n[9:] for n in df_q["Term"]]

In [ ]:
eps = 0.0005
neglogFDR = -np.log10(np.clip(df_q["FDR q-val"].values.astype(float), eps, 1.0))
df_q["-logFDR"] = neglogFDR

plot_df = df_q.sort_values("NES", ascending=False)

# colormap
cmap = matplotlib.colormaps.get_cmap("viridis")
if 0.01 < df_q["FDR q-val"].values.min():
    norm = matplotlib.colors.Normalize(vmin=-np.log10(0.05),vmax=-np.log10(0.01))
    ticks = -np.log10([0.05, 0.02, 0.01])
    tick_label = ["0.05", "0.02", "0.01"]
elif 0.001 <  df_q["FDR q-val"].values.min():
    norm = matplotlib.colors.Normalize(vmin=-np.log10(0.05),vmax=-np.log10(0.005))
    ticks = -np.log10([0.05, 0.02, 0.01, 0.005])
    tick_label = ["0.05", "0.02", "0.01", "0.005"]
else:
    norm = matplotlib.colors.Normalize(vmin=-np.log10(0.05),vmax=-np.log10(eps))
    ticks = -np.log10([0.05, 0.005, 0.0005])
    tick_label = ["0.05", "0.005", "0.0005"]
bar_colors = cmap(norm(plot_df["-logFDR"].values))


In [ ]:
ht = max(5, len(df_q))
tp, bt = 2, 2.5
fig, ax = plt.subplots(figsize=(6, (ht+tp+bt)*0.2))
fig.subplots_adjust(left=0.55, right=0.82, bottom=bt/(ht+tp+bt), top=1-tp/(ht+tp+bt))

ax.barh(
    y=plot_df["Term"],
    width=plot_df["NES"],
    color=bar_colors,
    edgecolor="none"
)

# lune x = 0
ax.axvline(0, linewidth=1)

ax.tick_params("y", labelsize=12)
ax.tick_params("x", labelsize=12)
ax.set_xlabel("NES", size=12)
ax.set_ylabel("")

ax.invert_yaxis()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# colorbar
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
c_ax = fig.add_axes([0.85, 1-(6+tp)/(ht+tp+bt), 0.03, 4.5/(ht+tp+bt)])
cbar = fig.colorbar(sm, cax=c_ax, extend="max", extendfrac=0.2)
cbar.set_ticks(ticks, labels=tick_label, fontsize=12)

c_ax.set_title("FDR q-val", fontsize=12, loc="center", y=1.1, x=1.5)
cbar.ax.invert_yaxis()

fig.suptitle(f"{dir_name} signal positive vs. negative\n{sub_d[nnn]}", fontsize=12)

plt.savefig(f"{fig_dir}nei2epi_signal_GSEA_{sub_d[nnn].replace("/","_")}_HALLMARK.png", dpi=500, format="png", bbox_inches=None)